# Train DBNet on Colab

Runtime → Change runtime type → **GPU** before running.

Expects the dataset on Google Drive at `MyDrive/dataset_receipt/` (`images/` + `metadata.pkl`).

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/dataset_receipt'
CKPT_DIR = '/content/drive/MyDrive/document-processing/checkpoints'  # on Drive so it survives disconnects

In [ ]:
import os

# For a private repo, add a GITHUB_TOKEN secret in the Colab sidebar (key icon). Never hardcode it here.
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None

auth = f"{token}@" if token else ""
REPO_URL = f"https://{auth}github.com/alexisvannson/document-processing.git"
REPO_DIR = '/content/document-processing'
BRANCH = 'main'

if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

In [ ]:
# Colab already ships torch/torchvision with CUDA; this installs the rest.
!pip install -q -r requirements.txt

In [ ]:
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())

In [ ]:
# Copy the dataset to local disk: reading hundreds of images from Drive every epoch is very slow.
!rsync -a --info=progress2 {DRIVE_DIR}/ /content/dataset_receipt/
!ls /content/dataset_receipt/images | wc -l

In [ ]:
EPOCHS = 300
BATCH_SIZE = 8
LR = 1e-3
NUM_WORKERS = 2  # Colab gives 2 CPU cores
VAL_EVERY = 10

!python train_dbnet.py \
    --metadata /content/dataset_receipt/metadata.pkl \
    --img-dir /content/dataset_receipt/images \
    --epochs {EPOCHS} --batch-size {BATCH_SIZE} --lr {LR} \
    --num-workers {NUM_WORKERS} --val-every {VAL_EVERY} \
    --out-dir {CKPT_DIR}

In [ ]:
!ls -lh {CKPT_DIR}

## Text recognition (ViT encoder → BERT decoder)

Trains on word crops cut from the ground-truth boxes. ~225M params at 384×384: lower `TROCR_BATCH_SIZE` if you hit CUDA OOM.

In [ ]:
TROCR_EPOCHS = 30
TROCR_BATCH_SIZE = 16
TROCR_LR = 5e-5
TROCR_DIR = '/content/drive/MyDrive/document-processing/checkpoints/trocr'

!python train_trocr.py \
    --metadata /content/dataset_receipt/metadata.pkl \
    --img-dir /content/dataset_receipt/images \
    --epochs {TROCR_EPOCHS} --batch-size {TROCR_BATCH_SIZE} --lr {TROCR_LR} \
    --num-workers 2 \
    --out-dir {TROCR_DIR}

In [ ]:
!ls -lh {TROCR_DIR}/best